In [1]:
import os
import cv2
import shutil
import albumentations as A
from pathlib import Path

src_img_dir = Path("CarDD_YOLO/images/train")
src_lbl_dir = Path("CarDD_YOLO/labels/train")

out_img_dir = Path("CarDD_YOLO_AUG/images/train")
out_lbl_dir = Path("CarDD_YOLO_AUG/labels/train")

shutil.copytree("CarDD_YOLO", "CarDD_YOLO_AUG", dirs_exist_ok=True)
yaml_path = Path("CarDD_YOLO_AUG/dataset.yaml")

if yaml_path.exists():
    text = yaml_path.read_text()

    text = text.replace(
        "/home/sagemaker-user/Project/TrainModel/CarDD_YOLO",
        "/home/sagemaker-user/Project/TrainModel/CarDD_YOLO_AUG"
    )

    yaml_path.write_text(text)

transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),

        A.OneOf(
            [
                A.RandomBrightnessContrast(
                    brightness_limit=0.25,
                    contrast_limit=0.25,
                    p=1.0,
                ),
                A.HueSaturationValue(
                    hue_shift_limit=10,
                    sat_shift_limit=25,
                    val_shift_limit=20,
                    p=1.0,
                ),
                A.CLAHE(
                    clip_limit=2.0,
                    tile_grid_size=(8, 8),
                    p=1.0,
                ),
            ],
            p=0.6,
        ),

        A.ShiftScaleRotate(
            shift_limit=0.08,
            scale_limit=0.35,
            rotate_limit=8,
            border_mode=cv2.BORDER_CONSTANT,
            value=(114, 114, 114),
            p=0.65,
        ),

        A.Perspective(
            scale=(0.02, 0.06),
            keep_size=True,
            pad_mode=cv2.BORDER_CONSTANT,
            pad_val=(114, 114, 114),
            p=0.25,
        ),

        A.OneOf(
            [
                A.MotionBlur(blur_limit=5, p=1.0),
                A.GaussianBlur(blur_limit=5, p=1.0),
                A.GaussNoise(var_limit=(10.0, 40.0), p=1.0),
            ],
            p=0.25,
        ),

        A.CoarseDropout(
            max_holes=6,
            max_height=32,
            max_width=32,
            min_holes=1,
            min_height=8,
            min_width=8,
            fill_value=(114, 114, 114),
            p=0.20,
        ),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"],
        min_visibility=0.25,
        clip=True,
        filter_invalid_bboxes=True,
    ),
)

def clip01(x):
    return max(0.0, min(1.0, x))

def read_yolo_labels(path):
    boxes = []
    labels = []

    if not path.exists():
        return boxes, labels

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            cls, x, y, w, h = parts
            x = clip01(float(x))
            y = clip01(float(y))
            w = clip01(float(w))
            h = clip01(float(h))

            if w <= 0 or h <= 0:
                continue

            labels.append(int(cls))
            boxes.append([x, y, w, h])

    return boxes, labels

def write_yolo_labels(path, boxes, labels):
    with open(path, "w") as f:
        for box, cls in zip(boxes, labels):
            x, y, w, h = box
            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

num_aug = 2

for img_path in src_img_dir.glob("*.jpg"):
    label_path = src_lbl_dir / f"{img_path.stem}.txt"

    image = cv2.imread(str(img_path))
    boxes, labels = read_yolo_labels(label_path)

    if len(boxes) == 0:
        continue

    for i in range(num_aug):
        augmented = transform(
            image=image,
            bboxes=boxes,
            class_labels=labels,
        )

        aug_img = augmented["image"]
        aug_boxes = augmented["bboxes"]
        aug_labels = augmented["class_labels"]

        if len(aug_boxes) == 0:
            continue

        new_name = f"{img_path.stem}_aug{i}"
        cv2.imwrite(str(out_img_dir / f"{new_name}.jpg"), aug_img)
        write_yolo_labels(out_lbl_dir / f"{new_name}.txt", aug_boxes, aug_labels)

print("done")

/opt/conda/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_8534/1267495695.py:41: UserWarning: Argument(s) 'value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(
/tmp/ipykernel_8534/1267495695.py:50: UserWarning: Argument(s) 'pad_mode, pad_val' are not valid for transform Perspective
  A.Perspective(
/tmp/ipykernel_8534/1267495695.py:62: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 40.0), p=1.0),
/tmp/ipykernel_8534/1267495695.py:67: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(


done


In [1]:
import os

print(len(os.listdir("CarDD_YOLO_AUG/images/train")))
print(len(os.listdir("CarDD_YOLO_AUG/images/val")))
print(len(os.listdir("CarDD_YOLO_AUG/images/test")))

5914
563
282


In [2]:
import os

print(len(os.listdir("CarDD_YOLO/images/train")))
print(len(os.listdir("CarDD_YOLO/images/val")))
print(len(os.listdir("CarDD_YOLO/images/test")))

1971
563
282
